## Landscape Classifier
### Scene Recognition with ResNet50

This notebook implements a deep learning model to classify landscape images using ResNet50.
The dataset contains images organized in class folders, and classes are automatically detected from the dataset directory structure.

In [ ]:
import os
import sys
import platform
import random
import numpy as np
import torch
import matplotlib.pyplot as plt

# Add scripts directory to path for custom imports
sys.path.insert(0, os.path.abspath('../scripts'))

# GPU & dataset utilities
from gpu_utils import CheckGPU, CheckCUDA, CheckGPUBrief
from dataset_counter import CountDataset

# Data pipeline
from data_utils import (
    build_transforms, load_and_split_dataset,
    compute_class_weights, create_dataloaders,
)

# Model building
from model_builder import (
    build_model, create_criterion, create_optimizer,
    create_scheduler, get_scheduler_params, save_checkpoint,
)

# Training
from trainer import train_model

# Evaluation
from evaluation import (
    run_test_inference, compute_confusion_metrics,
    compute_per_class_metrics, run_error_analysis,
)

# Reporting & visualization
from training_report import generate_training_report
from training_visualizer import plot_gradient_descent
from visualizer import (
    plot_sample_images, plot_training_curves,
    plot_confusion_matrices, plot_performance_analysis,
)

print("\u2705 All libraries and custom modules imported successfully!")

#### Detect GPU Available, Details, Cuda, and cuDNN

In [ ]:
# From scripts.gpu_utils
CheckGPU()
CheckCUDA()

### Global Configuration Variables

In [ ]:
# ============================
# GLOBAL CONFIGURATION
# ============================

# Dataset paths (folder-based dataset, no CSV)
DATASET_DIR = "../data/raw/Datasets2"

# Auto-derive dataset name from directory
DATASET_NAME = os.path.basename(os.path.normpath(DATASET_DIR))

# Train/Validation/Test split ratios (no pre-split, we split from the full dataset)
TRAIN_SPLIT = 0.70   # 70% for training
VAL_SPLIT = 0.15    # 15% for validation
TEST_SPLIT = 0.15    # 15% for testing

# ============================
# AUGMENTATION CONFIGURATION
# ============================
USE_AUGMENTATION = True

# Individual augmentation toggles (only applied when USE_AUGMENTATION = True)
AUGMENTATION_OPTIONS = {
    "random_horizontal_flip": True,     # Flip images horizontally
    "random_rotation": True,            # Rotate images by up to ROTATION_DEGREES
    "random_affine": True,              # Translation and scale jitter
    "color_jitter": True,               # Brightness, contrast, saturation, hue
    "random_perspective": True,          # Perspective distortion
    "random_resized_crop": True,         # Random crop with resize (scale variation)
    "gaussian_blur": True,              # Slight blur for robustness
    "random_erasing": True,             # Simulates occlusion
}

# Augmentation parameters
ROTATION_DEGREES = 20                    # Max rotation angle
AFFINE_TRANSLATE = (0.1, 0.1)           # Max translation (fraction of image)
AFFINE_SCALE = (0.85, 1.15)             # Scale range
COLOR_BRIGHTNESS = 0.3                   # Brightness jitter
COLOR_CONTRAST = 0.3                     # Contrast jitter
COLOR_SATURATION = 0.3                   # Saturation jitter
COLOR_HUE = 0.1                          # Hue jitter
PERSPECTIVE_DISTORTION = 0.2             # Perspective distortion scale
PERSPECTIVE_PROB = 0.5                   # Probability of perspective transform
RESIZED_CROP_SCALE = (0.8, 1.0)         # Random resized crop scale range
GAUSSIAN_BLUR_KERNEL = (3, 3)           # Gaussian blur kernel size
GAUSSIAN_BLUR_PROB = 0.3                # Probability of gaussian blur
ERASING_PROB = 0.1                       # Probability of random erasing
ERASING_SCALE = (0.02, 0.1)             # Erasing area scale range

# Weighted sampling for class imbalance
USE_WEIGHTED_SAMPLER = False

# Normalization values
# ImageNet pretrained values (recommended for transfer learning with ResNet50)
USE_IMAGENET_NORM = True
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

# Set normalization based on choice
if USE_IMAGENET_NORM:
    NORMALIZE_MEAN = IMAGENET_MEAN
    NORMALIZE_STD = IMAGENET_STD
    print("\U0001f4ca Using ImageNet normalization values (recommended for transfer learning)")
else:
    # Placeholder: compute from dataset if needed
    NORMALIZE_MEAN = IMAGENET_MEAN
    NORMALIZE_STD = IMAGENET_STD
    print("\U0001f4ca Using default normalization values")

# Image settings
IMG_HEIGHT = 224
IMG_WIDTH = 224

# Batch size
BATCH_SIZE = 32

# Model Architecture (fixed: ResNet50 for this assignment)
MODEL_ARCH = 'resnet50'

# Model save path
MODEL_SAVE_DIR = "../models"
os.makedirs(MODEL_SAVE_DIR, exist_ok=True)

# Seed for reproducibility
SEED = 42

# Set random seeds
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

print(f"\u2705 Global configuration set successfully!")
print(f"   Dataset: {DATASET_NAME} ({DATASET_DIR})")
print(f"   Model Architecture: {MODEL_ARCH.upper()}")
print(f"   Image Size: {IMG_HEIGHT}x{IMG_WIDTH}")
print(f"   Batch Size: {BATCH_SIZE}")
print(f"   Train/Val/Test Split: {TRAIN_SPLIT*100:.0f}% / {VAL_SPLIT*100:.0f}% / {TEST_SPLIT*100:.0f}%")
print(f"   Augmentation: {'ENABLED' if USE_AUGMENTATION else 'DISABLED'}")
print(f"   Weighted Sampler: {'ENABLED' if USE_WEIGHTED_SAMPLER else 'DISABLED'}")

### Dataset Analysis & Information

Load and analyze the landscape dataset structure, class distribution, and statistics.

In [ ]:
# Load dataset information using CountDataset utility
print("\U0001f50d Loading dataset information...")
dataset_info = CountDataset(DATASET_DIR)

# Extract class names and count
class_names = sorted([
    d for d in os.listdir(DATASET_DIR)
    if os.path.isdir(os.path.join(DATASET_DIR, d)) and not d.startswith('.')
])
NUM_CLASSES = len(class_names)

# Build class counts array (ordered by class name)
class_counts = np.array([dataset_info[cn]['samples'] for cn in class_names])
total_samples = int(dataset_info['total_samples'])

print(f"\n\U0001f4ca Class Distribution Analysis:")
print(f"   Total classes: {NUM_CLASSES}")
print(f"   Total samples: {total_samples:,}")
print(f"   Most populated class: {class_names[np.argmax(class_counts)]} ({class_counts.max():,} samples)")
print(f"   Least populated class: {class_names[np.argmin(class_counts)]} ({class_counts.min():,} samples)")
print(f"   Average per class: {class_counts.mean():.1f} samples")
print(f"   Imbalance ratio: {class_counts.max() / class_counts.min():.2f}x")

# Visualize class distribution
fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(class_names, class_counts, color='steelblue', alpha=0.8, edgecolor='black')
ax.axhline(y=class_counts.mean(), color='red', linestyle='--', linewidth=2, label=f'Average: {class_counts.mean():.0f}')
ax.set_xlabel('Class', fontsize=12)
ax.set_ylabel('Number of Samples', fontsize=12)
ax.set_title('Dataset Class Distribution', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

# Add count labels on bars
for bar, count in zip(bars, class_counts):
    ax.text(bar.get_x() + bar.get_width() / 2., bar.get_height() + 5,
            f'{count}', ha='center', va='bottom', fontweight='bold', fontsize=11)

plt.tight_layout()
plt.show()

print("\n\u2705 Dataset information loaded successfully!")

### Visualize Sample Images from Dataset

Display sample landscape images from each class to understand the data better.

In [ ]:
plot_sample_images(DATASET_DIR, class_names, NUM_CLASSES)

### Data Augmentation & Transforms

Define transforms for training (with configurable augmentation) and validation/test sets.
Landscape-specific augmentations optimized for scene recognition.

In [ ]:
train_transforms, val_test_transforms, applied_augmentations = build_transforms(
    img_height=IMG_HEIGHT,
    img_width=IMG_WIDTH,
    normalize_mean=NORMALIZE_MEAN,
    normalize_std=NORMALIZE_STD,
    use_augmentation=USE_AUGMENTATION,
    augmentation_options=AUGMENTATION_OPTIONS,
    rotation_degrees=ROTATION_DEGREES,
    affine_translate=AFFINE_TRANSLATE,
    affine_scale=AFFINE_SCALE,
    color_brightness=COLOR_BRIGHTNESS,
    color_contrast=COLOR_CONTRAST,
    color_saturation=COLOR_SATURATION,
    color_hue=COLOR_HUE,
    perspective_distortion=PERSPECTIVE_DISTORTION,
    perspective_prob=PERSPECTIVE_PROB,
    resized_crop_scale=RESIZED_CROP_SCALE,
    gaussian_blur_kernel=GAUSSIAN_BLUR_KERNEL,
    gaussian_blur_prob=GAUSSIAN_BLUR_PROB,
    erasing_prob=ERASING_PROB,
    erasing_scale=ERASING_SCALE,
)

### Load Dataset & Create Train/Val/Test Splits

Load the folder-based dataset using ImageFolder and split into train, validation, and test sets.

In [ ]:
data = load_and_split_dataset(
    dataset_dir=DATASET_DIR,
    train_split=TRAIN_SPLIT,
    val_split=VAL_SPLIT,
    test_split=TEST_SPLIT,
    train_transforms=train_transforms,
    val_test_transforms=val_test_transforms,
    seed=SEED,
)

full_dataset  = data["full_dataset"]
train_subset  = data["train_subset"]
val_subset    = data["val_subset"]
test_subset   = data["test_subset"]
train_dataset = data["train_dataset"]
val_dataset   = data["val_dataset"]
test_dataset  = data["test_dataset"]
NUM_CLASSES   = data["num_classes"]
class_names   = data["class_names"]
train_size    = data["train_size"]
val_size      = data["val_size"]
test_size     = data["test_size"]

### Compute Class Weights for Imbalanced Data

Calculate class weights to handle class imbalance in the dataset.
This helps the model learn minority classes better.

In [ ]:
weights_data = compute_class_weights(
    full_dataset=full_dataset,
    train_subset=train_subset,
    num_classes=NUM_CLASSES,
    class_names=class_names,
    use_weighted_sampler=USE_WEIGHTED_SAMPLER,
)

class_weights            = weights_data["class_weights"]
sample_weights           = weights_data["sample_weights"]
train_class_counts_array = weights_data["train_class_counts"]

### Create DataLoaders

Initialize PyTorch DataLoaders with appropriate batch size and sampling strategy.

In [ ]:
train_loader, val_loader, test_loader = create_dataloaders(
    train_dataset=train_dataset,
    val_dataset=val_dataset,
    test_dataset=test_dataset,
    batch_size=BATCH_SIZE,
    use_weighted_sampler=USE_WEIGHTED_SAMPLER,
    sample_weights=sample_weights,
)

#### PRINT SUMMARY

In [ ]:
print("\n" + "="*80)
print("\U0001f4ca PREPROCESSING SUMMARY")
print("="*80)
print(f"{'Dataset:':<30} {DATASET_NAME}")
print(f"{'Total Classes:':<30} {NUM_CLASSES}")
print(f"{'Class Names:':<30} {', '.join(class_names)}")
print(f"{'Training Samples:':<30} {train_size:,}")
print(f"{'Validation Samples:':<30} {val_size:,}")
print(f"{'Test Samples:':<30} {test_size:,}")
print(f"{'Total Samples:':<30} {total_samples:,}")
print(f"\n{'Image Processing:':<30}")
print(f"  {'- Target Size:':<28} {IMG_HEIGHT}x{IMG_WIDTH} pixels")
print(f"  {'- Normalization:':<28} {'ImageNet' if USE_IMAGENET_NORM else 'Custom'}")
print(f"\n{'Augmentation:':<30} {'ENABLED' if USE_AUGMENTATION else 'DISABLED'}")
if USE_AUGMENTATION:
    active = [k for k, v in AUGMENTATION_OPTIONS.items() if v]
    for aug_name in active:
        print(f"  - {aug_name}")
print(f"\n{'Class Balancing:':<30}")
print(f"  {'- Weighted Sampling:':<28} {'ENABLED' if USE_WEIGHTED_SAMPLER else 'DISABLED'}")
print(f"  {'- Class Imbalance Ratio:':<28} {train_class_counts_array.max() / train_class_counts_array.min():.2f}x")
print(f"\n{'Batch Configuration:':<30}")
print(f"  {'- Batch Size:':<28} {BATCH_SIZE}")
print(f"  {'- Train Batches/Epoch:':<28} {len(train_loader)}")
print(f"  {'- Val Batches/Epoch:':<28} {len(val_loader)}")
print(f"  {'- Test Batches:':<28} {len(test_loader)}")
print("="*80)
print("\u2705 Preprocessing complete! Ready for model training.\n")

#### MODEL TRAINING KEY VARIABLES

In [ ]:
# ============================
# MODEL TRAINING CONFIGURATION
# ============================

# Training hyperparameters
LEARNING_RATE = 1e-4          # Initial learning rate for AdamW
MAX_EPOCHS = 50               # Maximum training epochs
WEIGHT_DECAY = 1e-4           # L2 regularization to prevent overfitting
DROPOUT_RATE = 0.4            # Dropout in classifier head

# Early stopping
EARLY_STOPPING_PATIENCE = 8   # Stop if no improvement for 8 epochs
LABEL_SMOOTHING = 0.1         # Label smoothing for better generalisation (reduces overconfidence)

# Gradient clipping
MAX_GRAD_NORM = 1.0           # Prevent exploding gradients

# ============================
# LR SCHEDULER CONFIGURATION
# ============================
# Choose from: "CosineAnnealingWarmRestarts", "StepLR", "ExponentialLR",
#              "ReduceLROnPlateau", "CosineAnnealingLR"
LR_SCHEDULER = "ReduceLROnPlateau"

# --- CosineAnnealingWarmRestarts ---
COSINE_T_0 = 15               # Number of epochs for the first restart cycle
COSINE_T_MULT = 2             # Factor to increase cycle length after each restart
COSINE_ETA_MIN = 1e-7         # Minimum learning rate

# --- StepLR ---
STEP_SIZE = 10                # Decay LR every STEP_SIZE epochs
STEP_GAMMA = 0.1              # Multiplicative factor for LR decay

# --- ExponentialLR ---
EXP_GAMMA = 0.95              # Multiplicative factor per epoch

# --- ReduceLROnPlateau ---
PLATEAU_FACTOR = 0.5          # Factor to reduce LR
PLATEAU_PATIENCE = 3          # Epochs to wait before reducing LR
PLATEAU_MIN_LR = 1e-7         # Minimum LR

# --- CosineAnnealingLR ---
COSINE_T_MAX = 50             # Maximum number of iterations
COSINE_ANNEAL_ETA_MIN = 1e-7  # Minimum learning rate

# ============================
# PROGRESSIVE UNFREEZING
# ============================
USE_PROGRESSIVE_UNFREEZE = False   # Enable/disable progressive layer unfreezing

# Unfreezing schedule: at which epoch each layer group gets unfrozen
# Only used when USE_PROGRESSIVE_UNFREEZE = True
# Epochs are 1-indexed (epoch 1 = first epoch)
# Initially only the classifier head (model.fc) is trainable
UNFREEZE_SCHEDULE = {
    "layer4": 5,     # Unfreeze layer4 at epoch 5
    "layer3": 10,    # Unfreeze layer3 at epoch 10
    "layer2": 15,    # Unfreeze layer2 at epoch 15
    "layer1": 20,    # Unfreeze layer1 at epoch 20
}

# ============================
# MANUAL EARLY STOPPING
# ============================
# Press Ctrl+C (KeyboardInterrupt) during training to trigger manual early stop.
# The model will save the best weights found so far and exit gracefully.

# Model save directory
MODEL_SAVE_DIR = "../models"

# Scheduler display info (for use in summary cells)
_scheduler_info = {
    "CosineAnnealingWarmRestarts": f"CosineAnnealingWarmRestarts (T_0={COSINE_T_0}, T_mult={COSINE_T_MULT}, eta_min={COSINE_ETA_MIN})",
    "StepLR": f"StepLR (step_size={STEP_SIZE}, gamma={STEP_GAMMA})",
    "ExponentialLR": f"ExponentialLR (gamma={EXP_GAMMA})",
    "ReduceLROnPlateau": f"ReduceLROnPlateau (factor={PLATEAU_FACTOR}, patience={PLATEAU_PATIENCE}, min_lr={PLATEAU_MIN_LR})",
    "CosineAnnealingLR": f"CosineAnnealingLR (T_max={COSINE_T_MAX}, eta_min={COSINE_ANNEAL_ETA_MIN})",
}

print("\U0001f3af Training Configuration:")
print(f"   Model: {MODEL_ARCH.upper()}")
print(f"   Learning Rate: {LEARNING_RATE}")
print(f"   Weight Decay: {WEIGHT_DECAY}")
print(f"   Dropout Rate: {DROPOUT_RATE}")
print(f"   Max Epochs: {MAX_EPOCHS}")
print(f"   Early Stopping Patience: {EARLY_STOPPING_PATIENCE}")
print(f"   Gradient Clipping: {MAX_GRAD_NORM}")
print(f"   Label Smoothing: {LABEL_SMOOTHING}")
print(f"   LR Scheduler: {LR_SCHEDULER}")
print(f"   Progressive Unfreezing: {'ENABLED' if USE_PROGRESSIVE_UNFREEZE else 'DISABLED'}")
if USE_PROGRESSIVE_UNFREEZE:
    for layer, ep in sorted(UNFREEZE_SCHEDULE.items(), key=lambda x: x[1]):
        print(f"      - {layer}: unfreezes at epoch {ep}")
print(f"   Manual Early Stop: Ctrl+C during training")
print(f"   Classes: {NUM_CLASSES}")

### Load Pretrained Model Architecture

In [ ]:
# Build model using model_builder
CheckCUDA()
model, device, in_features = build_model(
    MODEL_ARCH, NUM_CLASSES, DROPOUT_RATE,
    use_progressive_unfreeze=USE_PROGRESSIVE_UNFREEZE,
)

#### LOSS & OPTIMIZER

In [ ]:
# Loss function, optimizer, and LR scheduler
criterion = create_criterion(class_weights, LABEL_SMOOTHING, device)
optimizer = create_optimizer(model, LEARNING_RATE, WEIGHT_DECAY)
scheduler = create_scheduler(
    optimizer, LR_SCHEDULER,
    cosine_t_0=COSINE_T_0, cosine_t_mult=COSINE_T_MULT, cosine_eta_min=COSINE_ETA_MIN,
    step_size=STEP_SIZE, step_gamma=STEP_GAMMA,
    exp_gamma=EXP_GAMMA,
    plateau_factor=PLATEAU_FACTOR, plateau_patience=PLATEAU_PATIENCE, plateau_min_lr=PLATEAU_MIN_LR,
    cosine_t_max=COSINE_T_MAX, cosine_anneal_eta_min=COSINE_ANNEAL_ETA_MIN,
)
print(f"\u2705 Full config: {_scheduler_info[LR_SCHEDULER]}")

#### Model Summary

In [ ]:
print("\n" + "="*60)
print("\U0001f680 MODEL SETUP COMPLETE")
print("="*60)
print(f"{'Architecture:':<25} {MODEL_ARCH.upper()}")
print(f"{'Input Size:':<25} {IMG_HEIGHT}x{IMG_WIDTH}")
print(f"{'Output Classes:':<25} {NUM_CLASSES} ({', '.join(class_names)})")
print(f"{'Total Parameters:':<25} {sum(p.numel() for p in model.parameters()):,}")
print(f"{'Trainable Parameters:':<25} {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
print(f"\n{'Training Settings:':<25}")
print(f"{'  Learning Rate:':<25} {LEARNING_RATE}")
print(f"{'  Weight Decay:':<25} {WEIGHT_DECAY}")
print(f"{'  Dropout Rate:':<25} {DROPOUT_RATE}")
print(f"{'  Max Epochs:':<25} {MAX_EPOCHS}")
print(f"{'  Early Stop Patience:':<25} {EARLY_STOPPING_PATIENCE}")
print(f"{'  Device:':<25} {device.type.upper()}")
print(f"{'  Loss Function:':<25} CrossEntropyLoss (weighted)")
print(f"{'  Optimizer:':<25} AdamW")
print(f"{'  LR Scheduler:':<25} {LR_SCHEDULER}")
print("="*60 + "\n")

### TRAINING & VALIDATION LOOP

In [ ]:
# Training functions (calculate_top_k_accuracy, _unfreeze_layer, train_model)
# have been moved to  src/scripts/trainer.py
CheckGPUBrief()

#### TRAIN THE MODEL & SAVE THE TRAINED MODEL

---

## Training Instructions

### Model: ResNet50

**Training Features:**
- Early stopping (patience=10 epochs)
- Overfitting detection and warnings
- Configurable LR scheduler (CosineAnnealingWarmRestarts, StepLR, ExponentialLR, ReduceLROnPlateau, CosineAnnealingLR)
- Gradient clipping
- Top-1 and Top-5 accuracy tracking
- Class-weighted loss for imbalanced data
- Dropout regularization
- Configurable data augmentation

### File Naming Examples:
- `Landscape_resnet50_E18_VAL92.45.pth` - ResNet50, epoch 18, 92.45% val acc

---

In [ ]:
# Train the model — history dict is created and returned by train_model()
model, best_epoch, best_val_acc, total_training_time, history = train_model(
    model=model,
    criterion=criterion,
    optimizer=optimizer,
    scheduler=scheduler,
    train_loader=train_loader,
    val_loader=val_loader,
    device=device,
    class_names=class_names,
    num_classes=NUM_CLASSES,
    num_epochs=MAX_EPOCHS,
    early_stopping_patience=EARLY_STOPPING_PATIENCE,
    max_grad_norm=MAX_GRAD_NORM,
    scheduler_name=LR_SCHEDULER,
    weight_decay=WEIGHT_DECAY,
    use_progressive_unfreeze=USE_PROGRESSIVE_UNFREEZE,
    unfreeze_schedule=UNFREEZE_SCHEDULE,
)

# Save model checkpoint
model_name, model_path = save_checkpoint(
    model=model,
    optimizer=optimizer,
    best_epoch=best_epoch,
    best_val_acc=best_val_acc,
    model_arch=MODEL_ARCH,
    num_classes=NUM_CLASSES,
    class_names=class_names,
    history=history,
    use_progressive_unfreeze=USE_PROGRESSIVE_UNFREEZE,
    unfreeze_schedule=UNFREEZE_SCHEDULE,
    save_dir=MODEL_SAVE_DIR,
    dataset_name=DATASET_NAME,
)

# Assemble training report data
import platform
scheduler_params = get_scheduler_params(
    LR_SCHEDULER,
    cosine_t_0=COSINE_T_0,       cosine_t_mult=COSINE_T_MULT,   cosine_eta_min=COSINE_ETA_MIN,
    step_size=STEP_SIZE,          step_gamma=STEP_GAMMA,
    exp_gamma=EXP_GAMMA,
    plateau_factor=PLATEAU_FACTOR, plateau_patience=PLATEAU_PATIENCE, plateau_min_lr=PLATEAU_MIN_LR,
    cosine_t_max=COSINE_T_MAX,    cosine_anneal_eta_min=COSINE_ANNEAL_ETA_MIN,
)

report_data = {
    "dataset_name":            DATASET_NAME,
    "model_arch":              MODEL_ARCH,
    "pretrained_weights":      "IMAGENET1K_V2",
    "img_size":                (IMG_HEIGHT, IMG_WIDTH),
    "num_classes":             NUM_CLASSES,
    "class_names":             class_names,
    "total_params":            sum(p.numel() for p in model.parameters()),
    "fc_in_features":          in_features,
    "dropout_rate":            DROPOUT_RATE,
    "dataset_dir":             DATASET_DIR,
    "total_samples":           len(full_dataset),
    "train_size":              train_size,
    "val_size":                val_size,
    "test_size":               test_size,
    "train_split":             TRAIN_SPLIT,
    "val_split":               VAL_SPLIT,
    "test_split":              TEST_SPLIT,
    "batch_size":              BATCH_SIZE,
    "use_weighted_sampler":    USE_WEIGHTED_SAMPLER,
    "use_augmentation":        USE_AUGMENTATION,
    "augmentation_options":    AUGMENTATION_OPTIONS,
    "rotation_degrees":        ROTATION_DEGREES,
    "optimizer_name":          "AdamW",
    "learning_rate":           LEARNING_RATE,
    "weight_decay":            WEIGHT_DECAY,
    "label_smoothing":         LABEL_SMOOTHING,
    "max_grad_norm":           MAX_GRAD_NORM,
    "scheduler_name":          LR_SCHEDULER,
    "scheduler_params":        scheduler_params,
    "cosine_t_0":              COSINE_T_0,
    "cosine_t_mult":           COSINE_T_MULT,
    "cosine_eta_min":          COSINE_ETA_MIN,
    "max_epochs":              MAX_EPOCHS,
    "early_stopping_patience": EARLY_STOPPING_PATIENCE,
    "progressive_unfreeze":    USE_PROGRESSIVE_UNFREEZE,
    "unfreeze_schedule":       UNFREEZE_SCHEDULE if USE_PROGRESSIVE_UNFREEZE else None,
    "seed":                    SEED,
    "device":                  str(device),
    "gpu_name":                torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",
    "gpu_memory_gb":           f"{torch.cuda.get_device_properties(0).total_memory / (1024**3):.1f}" if torch.cuda.is_available() else "N/A",
    "cuda_version":            torch.version.cuda if torch.version.cuda else "N/A",
    "cudnn_version":           str(torch.backends.cudnn.version()) if torch.backends.cudnn.is_available() else "N/A",
    "torch_version":           torch.__version__,
    "python_version":          platform.python_version(),
    "os_info":                 platform.platform(),
    "history":                 history,
    "best_epoch":              best_epoch,
    "best_val_acc":            best_val_acc,
    "total_training_time":     total_training_time,
    "model_filename":          model_name,
    "model_path":              model_path,
}

# Save results directory, markdown report, and training visualisation
model_base_name = model_name.replace(".pth", "")
results_dir     = os.path.join(MODEL_SAVE_DIR, "results", model_base_name)
os.makedirs(results_dir, exist_ok=True)
print(f"Results directory: {results_dir}")

report_path = os.path.join(results_dir, f"{model_base_name}_report.md")
generate_training_report(report_data, report_path)

vis_path = os.path.join(results_dir, f"{model_base_name}_gradient_descent.png")
plot_gradient_descent(history, best_epoch, save_path=vis_path)

print(f"Report and visualization saved in: {results_dir}")

#### Plotting Training Curves

In [ ]:
plot_training_curves(history, MODEL_ARCH)

### Confusion Matrix

A confusion matrix provides a more detailed view of classification results.
It shows how many times the model correctly predicted each class versus how many times it confused it with another.

---

## POST-PROCESSING & EVALUATION

Comprehensive evaluation of the trained model on the test set.

In [ ]:
# Run test inference
inference = run_test_inference(model, test_loader, device)
all_preds  = inference["all_preds"]
all_labels = inference["all_labels"]
all_probs  = inference["all_probs"]

# Compute confusion metrics
metrics = compute_confusion_metrics(all_preds, all_labels, all_probs, class_names)
cm                    = metrics["cm"]
cm_normalized         = metrics["cm_normalized"]
accuracy              = metrics["accuracy"]
balanced_acc          = metrics["balanced_acc"]
per_class_acc         = metrics["per_class_acc"]
most_confused_idx     = metrics["most_confused_idx"]
most_confused_value   = metrics["most_confused_value"]
correct_confidences   = metrics["correct_confidences"]
incorrect_confidences = metrics["incorrect_confidences"]
avg_correct_conf      = metrics["avg_correct_conf"]
avg_incorrect_conf    = metrics["avg_incorrect_conf"]

# Plot confusion matrices
plot_confusion_matrices(cm, cm_normalized, class_names, MODEL_ARCH)

# Print classification report
print("\n\U0001f4ca Classification Report:")
print(metrics["classification_report_str"])

### Per-Class Detailed Metrics

Breakdown of precision, recall, and F1-score for each landscape class.

In [ ]:
# Compute per-class metrics
pcm = compute_per_class_metrics(all_preds, all_labels, class_names, per_class_acc)
precision          = pcm["precision"]
recall             = pcm["recall"]
f1                 = pcm["f1"]
support            = pcm["support"]
sorted_by_f1       = pcm["class_metrics"]
macro_precision    = pcm["macro_precision"]
macro_recall       = pcm["macro_recall"]
macro_f1           = pcm["macro_f1"]
weighted_precision = pcm["weighted_precision"]
weighted_recall    = pcm["weighted_recall"]
weighted_f1        = pcm["weighted_f1"]

# Print per-class metrics table
print("\n\U0001f4ca Detailed Per-Class Metrics:")
print("="*90)
print(f"{'Class':<12} {'Accuracy':<12} {'Precision':<12} {'Recall':<12} {'F1-Score':<12} {'Support':<10}")
print("="*90)

print("\U0001f3c6 CLASSES RANKED BY F1-SCORE:")
for m in sorted_by_f1:
    print(f"{m['class']:<12} {m['accuracy']:>6.2f}%      {m['precision']:>6.2f}%      "
          f"{m['recall']:>6.2f}%      {m['f1']:>6.2f}%      {m['support']:<10}")
print("="*90)

print(f"\n\U0001f4ca AGGREGATE METRICS:")
print(f"  Macro Avg:    Precision={macro_precision:.2f}%  Recall={macro_recall:.2f}%  F1={macro_f1:.2f}%")
print(f"  Weighted Avg: Precision={weighted_precision:.2f}%  Recall={weighted_recall:.2f}%  F1={weighted_f1:.2f}%")

### Performance Visualizations

Additional charts for comprehensive model analysis.

In [ ]:
plot_performance_analysis(
    class_names, NUM_CLASSES, per_class_acc, accuracy,
    precision, recall, f1, support,
    correct_confidences, incorrect_confidences, cm, MODEL_ARCH,
)

### Error Analysis

Analyzing misclassification patterns to understand model weaknesses.

In [ ]:
# Run error analysis
errors = run_error_analysis(all_preds, all_labels, all_probs, class_names, support, incorrect_confidences)
num_errors = errors["num_errors"]

print(f"\u274c Total misclassifications: {num_errors} out of {len(all_labels)} ({errors['error_rate']:.2f}%)")

print(f"\n\U0001f534 Classes with Most Errors:")
print("="*60)
print(f"{'Class':<15} {'Errors':<12} {'Error Rate':<15}")
print("="*60)
for cls_idx, error_count in errors["sorted_error_classes"]:
    err_rate = (error_count / support[cls_idx]) * 100 if support[cls_idx] > 0 else 0
    print(f"   {class_names[cls_idx]:<15} {error_count:<12} {err_rate:>6.2f}%")
print("="*60)

if incorrect_confidences:
    q = errors["conf_quartiles"]
    print(f"\n\U0001f4ca Confidence for Misclassifications:")
    print(f"  25th percentile: {q[0]*100:.2f}%")
    print(f"  50th percentile (median): {q[1]*100:.2f}%")
    print(f"  75th percentile: {q[2]*100:.2f}%")
    print(f"  High-confidence errors (>80%): {errors['high_confidence_errors']}")

print(f"\n\U0001f504 Most Common Misclassification Patterns:")
print("="*70)
print(f"{'True Class':<15} {'Predicted As':<15} {'Count':<12} {'% of Class':<15}")
print("="*70)
for p in errors["top_confusion_patterns"]:
    print(f"   {p['true_class']:<15} {p['predicted_as']:<15} {p['count']:<12} {p['pct_of_class']:>6.2f}%")
print("="*70)

print("\n\u2705 Error analysis complete!")

### Final Summary Report

Comprehensive summary of model performance and key findings.

In [ ]:
# Final Summary Report
print("\n" + "="*80)
print(" "*25 + "\U0001f3af MODEL EVALUATION SUMMARY")
print("="*80)

print(f"\n\U0001f4e6 Model Architecture: {MODEL_ARCH.upper()}")
print(f"\U0001f4ca Dataset: {DATASET_NAME}")
print(f"\U0001f522 Number of Classes: {NUM_CLASSES} ({', '.join(class_names)})")
print(f"\U0001f9ea Test Set Size: {len(all_labels)} images")

print(f"\n{'\u2500'*80}")
print("\U0001f4c8 OVERALL PERFORMANCE METRICS:")
print(f"{'\u2500'*80}")
print(f"  Overall Accuracy:        {accuracy * 100:>6.2f}%")
print(f"  Balanced Accuracy:       {balanced_acc * 100:>6.2f}%")
print(f"  Macro Precision:         {macro_precision:>6.2f}%")
print(f"  Macro Recall:            {macro_recall:>6.2f}%")
print(f"  Macro F1-Score:          {macro_f1:>6.2f}%")
print(f"  Weighted Precision:      {weighted_precision:>6.2f}%")
print(f"  Weighted Recall:         {weighted_recall:>6.2f}%")
print(f"  Weighted F1-Score:       {weighted_f1:>6.2f}%")

print(f"\n{'\u2500'*80}")
print("\u2705 BEST PERFORMING CLASSES (Top-3):")
print(f"{'\u2500'*80}")
for i, metric in enumerate(sorted_by_f1[:3], 1):
    print(f"  {i}. {metric['class']}: F1={metric['f1']:6.2f}%, "
          f"Accuracy={metric['accuracy']:6.2f}%, Support={metric['support']}")

print(f"\n{'\u2500'*80}")
print("\u26a0\ufe0f  WORST PERFORMING CLASSES (Bottom-3):")
print(f"{'\u2500'*80}")
for i, metric in enumerate(sorted_by_f1[-3:], 1):
    print(f"  {i}. {metric['class']}: F1={metric['f1']:6.2f}%, "
          f"Accuracy={metric['accuracy']:6.2f}%, Support={metric['support']}")

print(f"\n{'\u2500'*80}")
print("\U0001f504 CONFUSION INSIGHTS:")
print(f"{'\u2500'*80}")
print(f"  Most confused pair: {class_names[most_confused_idx[0]]} \u2192 {class_names[most_confused_idx[1]]} "
      f"({most_confused_value} misclassifications)")
print(f"  Total misclassifications: {num_errors}")
print(f"  Error rate: {num_errors/len(all_labels)*100:.2f}%")

print(f"\n{'\u2500'*80}")
print("\U0001f3b2 CONFIDENCE STATISTICS:")
print(f"{'\u2500'*80}")
print(f"  Avg confidence (correct):   {avg_correct_conf * 100:>6.2f}%")
print(f"  Avg confidence (incorrect): {avg_incorrect_conf * 100:>6.2f}%")
print(f"  Confidence gap:             {(avg_correct_conf - avg_incorrect_conf) * 100:>6.2f}%")

print(f"\n{'\u2500'*80}")
print("\U0001f4be MODEL ARTIFACTS:")
print(f"{'\u2500'*80}")
print(f"  Saved model: {model_name}")
print(f"  Training history: Stored in 'history' dictionary")
print(f"  Metrics tracked: Top-1 Acc, Top-5 Acc, Loss, Learning Rate")

print(f"\n{'\u2500'*80}")
print("\U0001f680 TRAINING CONFIGURATION:")
print(f"{'\u2500'*80}")
print(f"  Optimizer: AdamW (lr={LEARNING_RATE}, weight_decay={WEIGHT_DECAY})")
print(f"  Scheduler: {_scheduler_info[LR_SCHEDULER]}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Max epochs: {MAX_EPOCHS}")
print(f"  Early stopping patience: {EARLY_STOPPING_PATIENCE}")
print(f"  Dropout rate: {DROPOUT_RATE}")
print(f"  Gradient clipping: max_norm={MAX_GRAD_NORM}")
print(f"  Train/Val/Test split: {TRAIN_SPLIT*100:.0f}%/{VAL_SPLIT*100:.0f}%/{TEST_SPLIT*100:.0f}%")
print(f"  Augmentation: {'ENABLED' if USE_AUGMENTATION else 'DISABLED'}")
print(f"  Weighted Sampler: {'ENABLED' if USE_WEIGHTED_SAMPLER else 'DISABLED'}")

print(f"\n{'='*80}")
print(" "*20 + "\u2705 EVALUATION COMPLETE!")
print("="*80 + "\n")